# RAG Evaluation Workshop

Learn how to evaluate RAG systems using various metrics and frameworks.

## Setup

In [ ]:
!pip install ragas datasets langchain-openai

## Prepare Evaluation Data

In [ ]:
from datasets import Dataset

# Sample evaluation data
eval_data = {
    "question": [
        "What is RAG?",
        "How does vector search work?",
        "What are the benefits of RAG?",
        "What is the difference between RAG and fine-tuning?",
        "When should you use RAG?"
    ],
    "answer": [
        "RAG stands for Retrieval-Augmented Generation...",
        "Vector search uses embeddings to find similar...",
        "RAG helps reduce hallucinations and provides...",
        "Fine-tuning adapts a model to specific tasks...",
        "You should use RAG when you need to query..."
    ],
    "contexts": [
        ["RAG is a framework...", "It combines retrieval..."],
        ["Vector embeddings...", "Similarity search..."],
        ["Benefits include...", "Reduced hallucinations..."],
        ["Fine-tuning is...", "RAG is..."],
        ["Use RAG for...", "External knowledge..."]
    ],
    "ground_truth": [
        "RAG is Retrieval-Augmented Generation, a technique...",
        "Vector search works by converting text to embeddings...",
        "RAG provides fresh knowledge, reduces hallucinations...",
        "Fine-tuning modifies model weights, RAG retrieves...",
        "Use RAG for question answering, knowledge bases..."
    ]
}

dataset = Dataset.from_dict(eval_data)
print(dataset)

## Evaluate with RAGAS

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

# Run evaluation
results = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ]
)

print(results)

## Custom LLM-as-Judge Evaluator

In [ ]:
from langchain_ollama import ChatOllama

class RAGEvaluator:
    """Evaluate RAG using LLM as judge."""
    
    def __init__(self, llm=None):
        self.llm = llm or ChatOllama(model="llama3.2")
    
    def evaluate_faithfulness(self, question, answer, context):
        prompt = f"""Rate faithfulness 0-10:\n\nQuestion: {question}\n\nContext: {context}\n\nAnswer: {answer}\n\nOnly respond with a number:"""
        return float(self.llm.invoke(prompt).content.strip()) / 10
    
    def evaluate_answer_quality(self, question, answer):
        prompt = f"""Rate answer quality 0-10:\n\nQuestion: {question}\n\nAnswer: {answer}\n\nOnly respond with a number:"""
        return float(self.llm.invoke(prompt).content.strip()) / 10
    
    def full_evaluation(self, question, answer, contexts):
        context_str = "\n\n".join(contexts)
        return {
            "faithfulness": self.evaluate_faithfulness(question, answer, context_str),
            "answer_quality": self.evaluate_answer_quality(question, answer)
        }

evaluator = RAGEvaluator()

# Evaluate all questions
for i in range(len(eval_data["question"])):
    result = evaluator.full_evaluation(
        eval_data["question"][i],
        eval_data["answer"][i],
        eval_data["contexts"][i]
    )
    print(f"Q{i+1}: Faithfulness={result['faithfulness']:.2f}, Quality={result['answer_quality']:.2f}")

## Retrieval Metrics

In [ ]:
def precision_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    return len(set(retrieved_k) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    return len(set(retrieved_k) & set(relevant)) / len(relevant)

# Example
retrieved = ["doc1", "doc2", "doc3", "doc4", "doc5"]
relevant = ["doc1", "doc3", "doc5"]

print(f"Precision@3: {precision_at_k(retrieved, relevant, 3):.2f}")
print(f"Recall@3: {recall_at_k(retrieved, relevant, 3):.2f}")
print(f"Precision@5: {precision_at_k(retrieved, relevant, 5):.2f}")
print(f"Recall@5: {recall_at_k(retrieved, relevant, 5):.2f}")

## Benchmarking Function

In [ ]:
import time

def benchmark_rag(rag_pipeline, test_queries):
    """Benchmark a RAG pipeline."""
    results = []
    
    for query in test_queries:
        start = time.time()
        response = rag_pipeline.invoke({"query": query})
        latency = (time.time() - start) * 1000
        
        results.append({
            "query": query,
            "latency_ms": latency,
            "answer": response.get("result", "")[:100]
        })
    
    # Summary
    avg_latency = sum(r["latency_ms"] for r in results) / len(results)
    
    return {
        "results": results,
        "avg_latency_ms": avg_latency,
        "total_queries": len(results)
    }

# Run benchmark
test_queries = [
    "What is RAG?",
    "How does it work?",
    "Benefits?"
]

# benchmark = benchmark_rag(qa_chain, test_queries)
# print(f"Average latency: {benchmark['avg_latency_ms']:.2f}ms")

## Next Steps

- Add more test questions
- Compare different RAG configurations
- Set up continuous evaluation
- Create evaluation dashboards